In [1]:
import pandas as pd
import numpy as np
import os
import time

# Analyse intensité carbonique industrielles en Belgique

## Objectif

Construire un outil d'analyse de l'**intensité carbonique** des sites industriels 
belges, à destination des agences régionales de l'environnement (AwAC, VMM, 
Bruxelles Environnement), pour suivre la trajectoire de décarbonation 
sur la période 2007-2024.

## Périmètre d'analyse

L'analyse s'appuie sur deux sources E-PRTR à des grains complémentaires :
- **F1_4 — Émissions par site** : CO2 et autres polluants déclarés au niveau 
  facility (toutes sources confondues : combustion + procédés industriels)
- **F5_2 — Énergie LCP** : consommation par combustible des grandes installations(cheminée) 
  de combustion (≥ 50 MW), unité réglementée par la directive IED

## Indicateur principal

**Intensité carbonique = CO2 émis (t) / énergie consommée (TJ)**

Calculé au niveau site (numérateur F1_4, dénominateur agrégé depuis F5_2).

## Chargement des tables

In [21]:
f1_path = "./User friendly .csv files/F1_4_Air_Releases_Facilities.csv"
f5_path = "./User friendly .csv files/F5_2_LCP_Energy_Emissions.csv"
f2_4_path = "./User friendly .csv files/F2_4_Water_Releases_Facilities.csv"
f3_2_path = "./User friendly .csv files/F3_2_Transfers_Facilities.csv"
f6_1_path="./User friendly .csv files/F6_1_IED_Installations.csv"

df_f1 = pd.read_csv(f1_path, low_memory=False)
df_f5 = pd.read_csv(f5_path, low_memory=False)
df_f2_4 = pd.read_csv(f2_4_path, low_memory=False)
df_f3_2 = pd.read_csv(f3_2_path, low_memory=False)
df_f6_1=pd.read_csv(f6_1_path,low_memory=False)

print("F1_4 shape :", df_f1.shape)
print("F5_2 shape :", df_f5.shape)
print("F2_4 shape :", df_f2_4.shape)
print("F3_2 shape :", df_f3_2.shape)
print("F6_1 shape :", df_f6_1.shape)

F1_4 shape : (372206, 16)
F5_2 shape : (401072, 14)
F2_4 shape : (254156, 16)
F3_2 shape : (65951, 15)
F6_1 shape : (477644, 31)


In [23]:
# Focus Belgique
be_f1 = df_f1[(df_f1["countryName"] == "Belgium") & (df_f1["Pollutant"] == "Carbon dioxide (CO2)")].copy()
be_f5 = df_f5[df_f5["countryName"] == "Belgium"].copy()
be_f2_4 = df_f2_4[df_f2_4["countryName"] == "Belgium"].copy()
be_f3_2 = df_f3_2[df_f3_2["countryName"] == "Belgium"].copy()
be_f6_1=df_f6_1[df_f6_1["CountryName"]=="Belgium"].copy()


print("Belgique - F1 CO2 :", be_f1.shape)
print("Belgique - F5 :", be_f5.shape)
print("Belgique - F2_4 :", be_f2_4.shape)
print("Belgique - F3_2 :", be_f3_2.shape)
print("Belgique - F6_1 :", be_f6_1.shape)

print("\nAnnées F1 CO2 :", sorted(be_f1["reportingYear"].dropna().unique()))
print("Années F5 :", sorted(be_f5["reportingYear"].dropna().unique()))
print("Années F2_4 :", sorted(be_f2_4["reportingYear"].dropna().unique()))
print("Années F3_2 :", sorted(be_f3_2["reportingYear"].dropna().unique()))
print("Année F6_1:", sorted(be_f3_2["reportingYear"].dropna().unique()))


Belgique - F1 CO2 : (1270, 16)
Belgique - F5 : (10521, 14)
Belgique - F2_4 : (7044, 16)
Belgique - F3_2 : (1175, 15)
Belgique - F6_1 : (20129, 31)

Années F1 CO2 : [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Années F5 : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Années F2_4 : [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Années F3_2 : [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64

## information des tables

**Liens entre les tables E-PRTR**

````
┌─────────────────────────────────────┐
│ F1_4_Air_Releases_Facilities        │
│ Grain : Site × Année × Polluant     │
│ PK : FacilityInspireId              │
└─────────────────┬───────────────────┘
                  │
                  │ 1 facility ──▶ N installations
                  │
                  ▲ parent_facilityInspireId
┌─────────────────┴───────────────────┐
│ F6_1_IED_Installations              │
│ Grain : Installation × Année        │
│ PK : InstallationInspireId          │
│ FK : parent_facilityInspireId       │
└─────────────────┬───────────────────┘
                  │
                  │ 1 installation ──▶ N LCP (?)
                  │ ⚠️ Clé de jointure à confirmer
                  │
                  ▲ ?
┌─────────────────┴───────────────────┐
│ F5_2_LCP_Energy_Emissions           │
│ Grain : LCP × Année × FeatureType   │
│ PK : LCPInspireId + reportingYear   │
│        + featureType                │
└─────────────────────────────────────┘
````

**Note** : la clé de jointure F6_1 ↔ F5_2 reste à confirmer.
Hypothèse à vérifier : décomposition de `LCPInspireId` permettant
de retrouver le `InstallationInspireId` parent.
````



### F5_2 — Consommation énergétique et émissions des LCP
**Grain : 1 LCP × 1 année × 1 type de mesure (combustible OU polluant OU caractéristique technique)**

In [7]:
df_f5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401072 entries, 0 to 401071
Data columns (total 14 columns):
 #   Column                                     Non-Null Count   Dtype  
---  ------                                     --------------   -----  
 0   PublicationDate                            401072 non-null  object 
 1   countryName                                401072 non-null  object 
 2   reportingYear                              401072 non-null  int64  
 3   LCPInspireId                               401072 non-null  object 
 4   installationPartName                       401072 non-null  object 
 5   installationPartNameConfidentialityReason  78 non-null      object 
 6   City_Of_Facility                           400994 non-null  object 
 7   addressConfidentialityReason               416 non-null     object 
 8   Longitude                                  401072 non-null  float64
 9   Latitude                                   401072 non-null  float64
 10  featureT

In [8]:
df_f5.head(5)

,PublicationDate,countryName,reportingYear,LCPInspireId,installationPartName,installationPartNameConfidentialityReason,City_Of_Facility,addressConfidentialityReason,Longitude,Latitude,featureType,unit,featureValue,confidentialityReason
0,2026/02/13,Germany,2017,https://registry.gdi-de.org/id/de.sn.sax4inspi...,Heizkraftwerk Leipzig Nord,NaN,Leipzig,NaN,12.377450,51.352070,LCPCharacteristics,MW,514.84,NaN
1,2026/02/13,Germany,2017,https://registry.gdi-de.org/id/de.nw.inspire.p...,Kessel 4 (Wasserrohr-Naturlaufkessel) (bivale...,NaN,Duisburg,NaN,6.697170,51.438040,LCPCharacteristics,MW,67.40,NaN
2,2026/02/13,Germany,2018,https://registry.gdi-de.org/id/de.sh/50083712_...,Heizkraftwerk Wedel,NaN,Wedel,NaN,9.724160,53.567000,LCPCharacteristics,MW,712.00,NaN
3,2026/02/13,Germany,2017,https://registry.gdi-de.org/id/de.by.inspire.p...,Heizkraftwerk Erlangen,NaN,Erlangen,NaN,11.002113,49.592519,LCPCharacteristics,MW,266.90,NaN
4,2026/02/13,Lithuania,2020,LT.CAED/110818317.PART2,Druskininkai district boiler,NaN,Druskininkai,NaN,23.986250,53.997513,LCPCharacteristics,MW,88.00,NaN


In [15]:
df_f5['featureType'].unique()

array(['LCPCharacteristics', 'DUST', 'SO2', 'NOX', 'CONFIDENTIAL',
       'Lignite', 'OtherGases', 'Peat', 'Coal', 'OtherSolidFuels',
       'NaturalGas', 'Biomass', 'LiquidFuels'], dtype=object)

### F1_4 — Émissions de polluants atmosphériques par site industriel
**Grain : 1 site (facility) × 1 année × 1 polluant**

In [16]:
df_f1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372206 entries, 0 to 372205
Data columns (total 16 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   PublicationDate               372206 non-null  object 
 1   countryName                   372206 non-null  object 
 2   reportingYear                 372206 non-null  int64  
 3   EPRTR_SectorCode              367221 non-null  float64
 4   EPRTR_SectorName              367221 non-null  object 
 5   EPRTRAnnexIMainActivity       367221 non-null  object 
 6   FacilityInspireId             372206 non-null  object 
 7   facilityName                  372205 non-null  object 
 8   city                          372073 non-null  object 
 9   Longitude                     371400 non-null  float64
 10  Latitude                      371400 non-null  float64
 11  addressConfidentialityReason  806 non-null     object 
 12  TargetRelease                 372206 non-nul

In [19]:
print(f"nombre de polluants référencés: {len(df_f1['Pollutant'].unique())}")

nombre de polluants référencés: 69


### Table F6_1_IED installation : lien entre table consommation énergétique et emission CO2

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477644 entries, 0 to 477643
Data columns (total 31 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   PublicationDate                        477644 non-null  object 
 1   CountryName                            477644 non-null  object 
 2   reportingYear                          477644 non-null  int64  
 3   parent_facilityInspireId               477644 non-null  object 
 4   EPRTRAnnexIMainActivity                415649 non-null  object 
 5   InstallationInspireId                  477644 non-null  object 
 6   installationName                       477644 non-null  object 
 7   installationNameConfidentialityReason  9368 non-null    object 
 8   City_of_Facility                       477349 non-null  object 
 9   Longitude                              477644 non-null  float64
 10  Latitude                               477644 non-null  

In [50]:
df_IED_installation.head()

,PublicationDate,CountryName,reportingYear,parent_facilityInspireId,EPRTRAnnexIMainActivity,InstallationInspireId,installationName,installationNameConfidentialityReason,City_of_Facility,Longitude,...,permitGranted,dateOfGranting,permitReconsidered,permitUpdated,dateOfLastUpdate,permitURL,RelevantChapter,BATConclusion,BATAEL,BATAELStricterCondition
0,2026/02/13,Germany,2023,https://registry.gdi-de.org/id/de.rp.inspire.p...,4(a)(ii),https://registry.gdi-de.org/id/de.rp.inspire.p...,Glykol-Fabrik,NaN,Ludwigshafen am Rhein,8.43055,...,Yes,1900-01-01,Yes,No,1900-01-01,https://sgdsued.rlp.de/service/genehmigungen-u...,NaN,LVOC,NaN,NaN
1,2026/02/13,Germany,2023,https://registry.gdi-de.org/id/de.rp.inspire.p...,4(a)(ii),https://registry.gdi-de.org/id/de.rp.inspire.p...,Spezialitäten-Fabrik,NaN,Ludwigshafen am Rhein,8.42603,...,Yes,1900-01-01,Yes,No,1900-01-01,https://sgdsued.rlp.de/service/genehmigungen-u...,NaN,CWW,NaN,NaN
2,2026/02/13,Germany,2023,https://registry.gdi-de.org/id/de.rp.inspire.p...,4(a)(ii),https://registry.gdi-de.org/id/de.rp.inspire.p...,Indol-Fabrik,NaN,Ludwigshafen am Rhein,8.43110,...,Yes,1900-01-01,Yes,No,1900-01-01,https://sgdsued.rlp.de/service/genehmigungen-u...,NaN,CWW,NaN,NaN
3,2026/02/13,Germany,2023,https://registry.gdi-de.org/id/de.rp.inspire.p...,4(a)(ii),https://registry.gdi-de.org/id/de.rp.inspire.p...,Primärreformer Ammoniak-IV,NaN,Ludwigshafen am Rhein,8.41466,...,Yes,1900-01-01,No,No,1900-01-01,NaN,ChapterIII,LCP,NaN,NaN
4,2026/02/13,Germany,2023,https://registry.gdi-de.org/id/de.rp.inspire.p...,4(a)(ii),https://registry.gdi-de.org/id/de.rp.inspire.p...,Salmiakgeist-Fabrik,NaN,Ludwigshafen am Rhein,8.42737,...,Yes,1900-01-01,No,No,1900-01-01,https://sgdsued.rlp.de/service/genehmigungen-u...,NaN,CWW,NaN,NaN


In [55]:
df_energy_emission.head()

,PublicationDate,countryName,reportingYear,LCPInspireId,installationPartName,installationPartNameConfidentialityReason,City_Of_Facility,addressConfidentialityReason,Longitude,Latitude,featureType,unit,featureValue,confidentialityReason
0,2026/02/13,Germany,2017,https://registry.gdi-de.org/id/de.sn.sax4inspi...,Heizkraftwerk Leipzig Nord,NaN,Leipzig,NaN,12.377450,51.352070,LCPCharacteristics,MW,514.84,NaN
1,2026/02/13,Germany,2017,https://registry.gdi-de.org/id/de.nw.inspire.p...,Kessel 4 (Wasserrohr-Naturlaufkessel) (bivale...,NaN,Duisburg,NaN,6.697170,51.438040,LCPCharacteristics,MW,67.40,NaN
2,2026/02/13,Germany,2018,https://registry.gdi-de.org/id/de.sh/50083712_...,Heizkraftwerk Wedel,NaN,Wedel,NaN,9.724160,53.567000,LCPCharacteristics,MW,712.00,NaN
3,2026/02/13,Germany,2017,https://registry.gdi-de.org/id/de.by.inspire.p...,Heizkraftwerk Erlangen,NaN,Erlangen,NaN,11.002113,49.592519,LCPCharacteristics,MW,266.90,NaN
4,2026/02/13,Lithuania,2020,LT.CAED/110818317.PART2,Druskininkai district boiler,NaN,Druskininkai,NaN,23.986250,53.997513,LCPCharacteristics,MW,88.00,NaN


### Données énergie pour la belgique

In [ ]:
# 1 ligne = 1 LCP × 1 année × 1 mesure

print(df_energy_emission[df_energy_emission['countryName']=='Belgium'].shape)

(10521, 14)


In [ ]:
mask_belgique= df_energy_emission['countryName']=='Belgium'
df_be_energy_emission = df_energy_emission[mask_belgique]

print(f'consommation énergie: {df_be_energy_emission['featureType'].unique()}')
print(f'unités énergie:{df_be_energy_emission['unit'].unique()}')
print(f'rapports disponibles:{df_be_energy_emission['reportingYear'].unique()}')

consommation énergie: ['LCPCharacteristics' 'NOX' 'DUST' 'SO2' 'NaturalGas' 'LiquidFuels'
 'OtherGases' 'Biomass' 'Lignite' 'Coal' 'OtherSolidFuels' 'Peat']
unités énergie:['MW' 'hours' 'tonnes' 'TJ']
rapports disponible:[2018 2016 2021 2020 2024 2017 2019 2022 2023]


**Table emission énergie** : une ligne = 1 LCP × 1 année × 1 mesure.

La colonne `featureType` contient trois types de mesures distincts :
- **Combustibles** (NaturalGas, Coal, Lignite, Biomass, LiquidFuels, OtherGases, OtherSolidFuels, Peat) : énergie consommée, exprimée en TJ dans `featureValue`
- **Polluants** (NOX, DUST, SO2) : émissions, exprimées en tonnes
- **Caractéristiques techniques** (LCPCharacteristics) : puissance installée (MW) ou heures de fonctionnement

**Note méthodologique** :
- Pour obtenir l'énergie totale consommée par un LCP sur une année, il faudra **agréger** (somme) les valeurs des `featureType` correspondant aux combustibles.
- Le CO2 n'est **pas déclaré directement** dans la table. Il faudra l'estimer en appliquant des **facteurs d'émission GIEC** propres à chaque combustible (calcul prévu en phase Silver).

In [62]:
print(df_be_energy_emission.groupby(['featureType', 'unit']).size())

featureType         unit  
Biomass             TJ        805
Coal                TJ        805
DUST                tonnes    816
LCPCharacteristics  MW        816
                    hours     816
Lignite             TJ        805
LiquidFuels         TJ        805
NOX                 tonnes    816
NaturalGas          TJ        806
OtherGases          TJ        805
OtherSolidFuels     TJ        805
Peat                TJ        805
SO2                 tonnes    816
dtype: int64


### Nombre de sites industriels en belgique

In [80]:
mask_CO2=df_emission_facilities['Pollutant']=='Carbon dioxide (CO2)'
mask_be_emission=df_emission_facilities['countryName']=='Belgium'
df_be_emission_CO2=df_emission_facilities[mask_CO2 & mask_be_emission]


# Combien de sites ont des données complètes sur plusieurs années ?
sites_par_annee = df_be_emission_CO2.groupby('facilityName')['reportingYear'].nunique()
print(f"Nombre de site ayant un rapport annuel d'emission: {sites_par_annee.sort_values(ascending=False).shape[0]}")


Nombre de site ayant un rapport annuel d'emission: 105


In [73]:
# Répartition - combien d'années par site ?
print(sites_par_annee.value_counts().sort_index())

# Les sites les plus complets
print(sites_par_annee.sort_values(ascending=False).shape)

reportingYear
1      6
2      7
3      3
4      5
5      3
6      4
7      4
8      2
9      1
10     4
11     4
12     1
13     5
14     3
15     5
16    10
17     3
18    35
Name: count, dtype: int64
(105,)


In [74]:
sites_complets = sites_par_annee[sites_par_annee == 18].index
print(df_emission_facilities[df_emission_facilities['facilityName'].isin(sites_complets)]['facilityName'].unique())

['VYNOVA BELGIUM' 'UMICORE - HOBOKEN' 'BAYER AGRICULTURE'
 'ARCELORMITTAL BELGIUM - GENT'
 'CARRIERES ET FOURS A CHAUX DUMONT WAUTIER' 'BOREALIS KALLO'
 'TotalEnergies Olefins Antwerp' 'APERAM STAINLESS BELGIUM'
 "ELECTRABEL - CENTRALE D'AMERCOEUR" 'ELECTRABEL CENTRALE RODENHUIZE'
 'TotalEnergies Refinery Antwerp'
 'ELECTRABEL - CENTRALE BAUDOUR/St GHISLAIN' 'CARMEUSE - Site de Moha'
 "HEIDELBERG MATERIALS BENELUX s.a. - Site d'Antoing" 'VPK Paper'
 'AIR LIQUIDE LARGE INDUSTRY' "HOLCIM Belgique - Usine d'OBOURG"
 'BASF ANTWERPEN' 'HEIDELBERG MATERIALS BENELUX s.a. - Site de Lixhe'
 'ELECTRABEL - CENTRALE DROGENBOS' 'BRUXELLES ENERGIE'
 'EXXONMOBIL PETROLEUM & CHEMICAL - ESSO RAFFINADERIJ' 'BURGO ARDENNES'
 'LUMINUS' 'AGC GLASS EUROPE - Site de Moustier'
 'LHOIST INDUSTRIE - Site de On' "IPALLE - Usine d'incinération"
 'STORA ENSO LANGERBRUGGE' 'INEOS OXIDE UTILITIES'
 "CARMEUSE - Site d'Aisemont" 'INEOS Aromatics Belgium'
 'TEREOS STARCH & SWEETENERS BELGIUM' 'ELECTRABEL CENTRALE HERDE

### Données d'emission CO2 au niveau européen

In [ ]:
# Combien de sites par pays ?
print(df_emission_facilities[mask_CO2].groupby('countryName')['facilityName'].nunique().sort_values(ascending=False))

countryName
Germany           571
United Kingdom    398
Italy             338
Spain             302
France            265
Poland            202
Netherlands       125
Sweden            121
Denmark           115
Finland           109
Belgium           105
Czechia            89
Norway             78
Portugal           75
Romania            74
Switzerland        69
Hungary            59
Austria            55
Greece             52
Bulgaria           48
Slovakia           41
Ireland            24
Estonia            20
Croatia            18
Lithuania          18
Latvia              9
Slovenia            8
Luxembourg          8
Iceland             6
Cyprus              5
Serbia              4
Malta               3
Name: facilityName, dtype: int64
countryName
Austria           18
Belgium           18
Bulgaria          18
Estonia           18
Denmark           18
Germany           18
France            18
Finland           18
Hungary           18
Ireland           18
Italy             18
Greece  

In [76]:
# Combien d'années couvertes par pays ?
print(df_emission_facilities[mask_CO2].groupby('countryName')['reportingYear'].nunique().sort_values(ascending=False))

countryName
Austria           18
Belgium           18
Bulgaria          18
Estonia           18
Denmark           18
Germany           18
France            18
Finland           18
Hungary           18
Ireland           18
Italy             18
Greece            18
Netherlands       18
Poland            18
Luxembourg        18
Latvia            18
Spain             18
Sweden            18
Slovenia          18
Romania           18
Portugal          18
Cyprus            17
Czechia           16
Iceland           16
Lithuania         16
Switzerland       16
Slovakia          16
United Kingdom    13
Malta             13
Croatia           11
Norway            11
Serbia             2
Name: reportingYear, dtype: int64


In [82]:
# Inspection des LCPInspireId belges
df_be_lcp = df_energy_emission[df_energy_emission['countryName']=='Belgium']
print(df_be_lcp['LCPInspireId'].head(10).tolist())

# Et comparer avec les InstallationInspireId belges de F6_1
df_be_ied = df_IED_installation[df_IED_installation['CountryName']=='Belgium']
print(df_be_ied['InstallationInspireId'].head(10).tolist())


['https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000069.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000060.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000067.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000054.PART', 'BE.WA/095010101.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000036.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000021.PART', 'BE.WA/225010101.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000013.PART', 'BE.WA/049010701.PART']
['BE.WA/297010100.INSTALLATION', 'BE.WA/099010100.INSTALLATION', 'BE.WA/047010100.INSTALLATION', 'BE.WA/240010100.INSTALLATION', 'BE.WA/010010400.INSTALLATION', 'BE.WA/029010300.INSTALLATION', 'BE.WA/130010